In [1]:
import tensorflow as tf
import keras
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.16.1
Keras: 3.4.1


In [2]:
# Force reload of latest version
%load_ext autoreload
%autoreload 2

# Core imports from your .py files
from feature_backbone import main as pretrain_main
from image_reconstructor import main as train_decoder_main
from log_explain_wandb_bkp import explain_main
from utils import (
    L2Normalization, build_encoder, build_image_reconstructor,
    build_classifier_from_encoder, find_last_conv_in_model,
    WandbEpochLogger, get_image_datasets, warmup_model, load_model
)
from label_predictor import main as finetune_main

import argparse
import types

In [3]:
# Check wandb installation
try:
    import wandb
    print("wandb module path:", wandb.__file__)
    print("wandb installed successfully")
except ImportError:
    print("wandb not installed. Install with: pip install wandb")
    print("Continuing without wandb logging...")

wandb module path: /Users/bhoopeshnsingh/mlenv/lib/python3.10/site-packages/wandb/__init__.py
wandb installed successfully


In [4]:
import wandb
print("Current wandb run:", wandb.run if 'wandb' in dir() else None)

Current wandb run: None


In [ ]:
# Initialize wandb run (optional - set to None to disable logging)
RUN_ID = "self-supervised-22-nov"

if 'wandb' in dir() and wandb.run is None:
    wandb.init(
        project="Explainable-Classification",
        name=RUN_ID,
        resume="allow"
    )
else:
    print("Wandb not available or already initialized")

In [ ]:
# Define base arguments (common to all tasks)
BASE_ARGS = dict(
    train_dir="../data/train",
    val_dir="../data/val",
    img_size=128,
    batch_size=32,
    output_dir="../saved_models",
    wandb_project="Explainable-Classification"
)

print("Base configuration:")
for k, v in BASE_ARGS.items():
    print(f"  {k}: {v}")

## 1. Self-Supervised Pretraining (SimCLR)

Train encoder using contrastive learning with NT-Xent loss.

In [ ]:
from feature_backbone import make_simclr_dataset, build_simclr_encoder, train_simclr, save_encoder

# Pretraining arguments
pretrain_args = types.SimpleNamespace(
    **BASE_ARGS,
    projection_dim=128,
    epochs=10,
    steps_per_epoch=200,
    lr=1e-4,
    temperature=0.1
)

# Create SimCLR dataset
simclr_ds, train_ds = make_simclr_dataset(
    pretrain_args.train_dir,
    pretrain_args.val_dir,
    pretrain_args.img_size,
    pretrain_args.batch_size
)

In [ ]:
# Build SimCLR encoder with projection head
encoder_vec = build_simclr_encoder(
    input_shape=(pretrain_args.img_size, pretrain_args.img_size, 3),
    projection_dim=pretrain_args.projection_dim,
    return_spatial=False
)
#print("Encoder architecture:")
#encoder_vec.summary()

In [ ]:
# Train the SimCLR encoder
print("\n" + "="*80)
print("TRAINING SIMCLR ENCODER")
print("="*80)

encoder = train_simclr(
    encoder_vec,
    simclr_ds,
    pretrain_args,
    wandb=wandb if 'wandb' in dir() and wandb.run else None
)

print("\nSimCLR training complete!")

In [ ]:
# Save the trained encoder
encoder_path = save_encoder(
    encoder,
    pretrain_args,
    "feature_backbone_spatial.keras",
    wandb=wandb if 'wandb' in dir() and wandb.run else None
)

print(f"Encoder saved to: {encoder_path}")

In [ ]:
# Clear memory
import gc
import tensorflow as tf

gc.collect()
tf.keras.backend.clear_session()
print("Memory cleared")

In [ ]:
from utils import load_model, warmup_model
from feature_backbone import log_details

# Load encoder
encoder = load_model(
    f"{pretrain_args.output_dir}/feature_backbone_spatial.keras",
    compile=False
)

first_input = encoder.input[0]           # pick the main image input
first_shape = tuple(
    dim for dim in first_input.shape if dim is not None
)

print("Warmup using:", first_shape)

encoder = warmup_model(encoder, first_shape)

#encoder = warmup_model(encoder, (pretrain_args.img_size, pretrain_args.img_size, 3))

# Log encoder details
last_conv, embedding = log_details(
    encoder,
    train_ds,
    wandb if 'wandb' in dir() and wandb.run else None
)

print(f"\nEncoder loaded successfully")
print(f"Last conv layer: {last_conv.name if last_conv else 'None'}")
print(f"Embedding shape: {embedding.shape}")

## 2. Image Reconstruction (Decoder Training)

Train decoder to reconstruct images from encoder spatial features.

In [ ]:
from image_reconstructor import train_image_reconstructor, evaluate_image_reconstructor
import types

# Reconstruction arguments
recon_args = types.SimpleNamespace(
    train_dir="../data/train",
    val_dir="../data/val",
    encoder_path="../saved_models/feature_backbone_spatial.keras",
    output_dir="../saved_models",
    img_size=128,
    batch_size=16,
    epochs=10,
    lr=1e-4,
    wandb_project="Explainable-Classification"
)

print("Reconstruction configuration:")
for k, v in vars(recon_args).items():
    print(f"  {k}: {v}")

In [ ]:
 # Train decoder
print("\n" + "="*80)
print("TRAINING IMAGE RECONSTRUCTOR")
print("="*80)

decoder, autoencoder = train_image_reconstructor(recon_args)

print("\n✅ Decoder training complete!")

In [ ]:
# Load datasets for evaluation
train_ds, val_ds, class_names = get_image_datasets(
    recon_args.train_dir,
    recon_args.val_dir,
    (recon_args.img_size, recon_args.img_size),
    recon_args.batch_size
)

print(f"Loaded datasets")
print(f"Classes: {class_names}")

In [ ]:
# Evaluate reconstruction quality
evaluate_image_reconstructor(
    recon_args.encoder_path,
    f"{recon_args.output_dir}/image_reconstructor.keras",
    val_ds,
    n=5,
    log_to_wandb=True if 'wandb' in dir() and wandb.run else False
)

print("\nReconstruction evaluation complete")

## 3. Classifier Training (Supervised Fine-tuning)

Fine-tune classifier in two stages:
1. Train head with frozen encoder
2. Fine-tune entire model with lower learning rate

In [ ]:
from label_predictor import (
    finetune_main, prepare_datasets, train_classifier_head,
    fine_tune_classifier
)
import types

# Classifier arguments
predictor_args = types.SimpleNamespace(
    train_dir="../data/train",
    val_dir="../data/val",
    encoder_path="../saved_models/feature_backbone_spatial.keras",
    output_dir="../saved_models",
    img_size=128,
    batch_size=16,
    num_classes=len(class_names),
    epochs_head=5,
    epochs_full=10,
    lr_head=1e-3,
    lr_full=1e-5,
    dropout=0.3,
    dense_units=256,
    wandb_project="Explainable-Classification"
)

print("Classifier configuration:")
for k, v in vars(predictor_args).items():
    print(f"  {k}: {v}")

In [ ]:
from utils import load_model
from label_predictor import train_classifier_head, fine_tune_classifier, evaluate_classifier
import wandb

# Load encoder
encoder = load_model(predictor_args.encoder_path, compile=False)
print(f"Loaded encoder from {predictor_args.encoder_path}")

In [ ]:
# Stage 1: Train classifier head (frozen encoder)
print("\n" + "="*80)
print("STAGE 1: TRAINING CLASSIFIER HEAD (FROZEN ENCODER)")
print("="*80)

clf = train_classifier_head(
    encoder,
    train_ds,
    val_ds,
    len(class_names),
    predictor_args.lr_head,
    predictor_args.epochs_head,
    predictor_args.output_dir
)

print("\nStage 1 complete!")

In [ ]:
# Stage 2: Fine-tune entire classifier (unfrozen encoder)
print("\n" + "="*80)
print("STAGE 2: FINE-TUNING ENTIRE CLASSIFIER")
print("="*80)

clf = fine_tune_classifier(
    encoder,
    clf,
    train_ds,
    val_ds,
    predictor_args.lr_full,
    predictor_args.epochs_full,
    predictor_args.output_dir
)

print("\nStage 2 complete!")

In [ ]:
# Save final classifier
import os

clf_path = os.path.join(predictor_args.output_dir, "classifier.keras")
clf.save(clf_path)
print(f"Classifier saved to: {clf_path}")

In [ ]:
# Log classifier details
from label_predictor import log_classifier_info

log_classifier_info(clf, train_ds, predictor_args.output_dir)
print("Classifier details logged")

## 4. Classifier Evaluation

Evaluate classifier performance on validation set.

In [ ]:
# Evaluate classifier
print("\n" + "="*80)
print("EVALUATING CLASSIFIER")
print("="*80)

evaluate_classifier(
    clf,
    val_ds,
    class_names,
    wandb_run=RUN_ID if 'wandb' in dir() and wandb.run else None
)

print("\n Evaluation complete")

## 5. Explainability & Visualization

Generate explainability visualizations:
- **Grad-CAM**: Attention heatmaps
- **Parts Discovery**: Unsupervised semantic segmentation
- **Reconstruction**: Image reconstruction quality

In [ ]:
import types
from log_explain_wandb_bkp import explain_main

# Explainability arguments
explain_args = types.SimpleNamespace(
    encoder_path="../saved_models/feature_backbone_spatial.keras",
    decoder_path="../saved_models/image_reconstructor.keras",
    classifier_path="../saved_models/classifier.keras",
    train_dir="../data/train",
    val_dir="../data/val",
    img_size=128,
    batch_size=16,
    n_parts=4,
    samples_per_class=5,
    class_names=",".join(class_names),
    wandb_project="Explainable-Classification"
)

print("Explainability configuration:")
for k, v in vars(explain_args).items():
    print(f"  {k}: {v}")

In [ ]:
# Run complete explainability pipeline
print("\n" + "="*80)
print("GENERATING EXPLAINABILITY VISUALIZATIONS")
print("="*80)

explain_main(
    explain_args.encoder_path,
    explain_args.decoder_path,
    explain_args.classifier_path,
    explain_args
)

print("\n Explainability visualizations complete!")

## 6. Manual Testing & Visualization

Test individual components on sample images.

In [ ]:
from log_explain_wandb_bkp import make_gradcam_heatmap
from utils import find_last_conv_in_model, load_model
import numpy as np
import matplotlib.pyplot as plt

# Load classifier
clf = load_model(explain_args.classifier_path, compile=False)

# Find last conv layer
last_conv = find_last_conv_in_model(clf)
print(f"Using conv layer: {last_conv.name if last_conv else 'None'}")

# Get a sample image
for images, labels in val_ds.take(1):
    img = images[0]
    label = labels[0]
    
    # Prepare input
    img_array = tf.expand_dims(img, 0) / 255.0
    
    # Generate Grad-CAM
    heatmap = make_gradcam_heatmap(
        img_array,
        clf,
        last_conv
    )
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.numpy().astype('uint8'))
    axes[0].set_title(f"Original (Label: {class_names[label.numpy()]})")
    axes[0].axis('off')
    
    axes[1].imshow(img.numpy().astype('uint8'))
    axes[1].imshow(heatmap, cmap='jet', alpha=0.4)
    axes[1].set_title("Grad-CAM Heatmap")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    break

print("\nGrad-CAM visualization complete")

In [ ]:
from parts_discovery import parts_discovery
import matplotlib.pyplot as plt
import numpy as np

# Get a sample image
for images, labels in val_ds.take(1):
    img = images[0].numpy()
    label = labels[0].numpy()
    
    # Discover parts
    parts_map = parts_discovery(
        img,
        clf,
        last_conv_layer=last_conv.name if last_conv else None,
        n_parts=4
    )
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.astype('uint8'))
    axes[0].set_title(f"Original (Label: {class_names[label]})")
    axes[0].axis('off')
    
    axes[1].imshow(parts_map, cmap='tab10')
    axes[1].set_title("Parts Discovery (4 parts)")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    break

print("\nParts discovery visualization complete")

## Summary

### Training Pipeline Complete! ✅

**Models trained:**
1. ✅ Self-supervised encoder (SimCLR)
2. ✅ Image reconstructor (Decoder)
3. ✅ Supervised classifier

**Explainability methods:**
1. ✅ Grad-CAM attention heatmaps
2. ✅ Parts discovery (unsupervised segmentation)
3. ✅ Image reconstruction

**Saved models:**
- `./saved_models/feature_backbone_spatial.keras`
- `./saved_models/image_reconstructor.keras`
- `./saved_models/classifier.keras`

All models and visualizations are logged to Weights & Biases (if enabled).

In [ ]:
# Close wandb run
if 'wandb' in dir() and wandb.run:
    wandb.finish()
    print("✅ Wandb run closed")
else:
    print("No active wandb run to close")